# 05 · Custom signal radar

Detect material changes between two YUCLAW composite signals and build a simple watchlist-style alert in pure Python.

> **Disclaimer.** Research and education only. Not investment advice. Signal labels are research classifications, not buy/sell recommendations. YUCLAW is not a registered investment adviser. Past results — backtested or forward-tracked — do not predict future performance.

## Compare two snapshots

We use `client.replay` to get two point-in-time snapshots for the same ticker and compare them.

In [ ]:
import yuclaw_py
client = yuclaw_py.Client()

earlier = client.replay('AMD', '2026-03-01')
later   = client.replay('AMD', '2026-05-13')

print(f"AMD on 2026-03-01: {earlier['label']:<14s}  score={earlier['score']:+.3f}")
print(f"AMD on 2026-05-13: {later['label']:<14s}  score={later['score']:+.3f}")
print(f"Δ score: {later['score'] - earlier['score']:+.3f}")

## Simple alert function

In [ ]:
def changed(t, d1, d2, threshold=0.15):
    a = client.replay(t, d1)
    b = client.replay(t, d2)
    delta = b['score'] - a['score']
    label_flip = a['label'] != b['label']
    return {
        'ticker': t,
        'flipped_label': label_flip,
        'delta_score': round(delta, 3),
        'material': label_flip or abs(delta) >= threshold,
        'from': (a['label'], round(a['score'], 3)),
        'to':   (b['label'], round(b['score'], 3)),
    }

for t in ['AMD', 'NVDA', 'INTC']:
    r = changed(t, '2026-03-01', '2026-05-13')
    flag = '⚠ ' if r['material'] else '  '
    print(f"{flag}{t}: {r['from']}  →  {r['to']}  (Δ {r['delta_score']:+.3f})")


In production this lives in `v3.radar.run` — the SDK version is for experimentation. The production radar adds: Telegram/Email/Slack adapters, an audit log, and the locked not-advice footer on every broadcast.

---
> **Disclaimer.** Research and education only. Not investment advice. Signal labels are research classifications, not buy/sell recommendations. YUCLAW is not a registered investment adviser. Past results — backtested or forward-tracked — do not predict future performance.